# 1. Download Dataset & Checksum

In [8]:
import urllib.request
import hashlib
from pathlib import Path

raw_dir = Path('../data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)

datasets = {
    "taxi_zone_lookup.csv": "https://d37ci6vzurychx.cloudfront.net/misc/taxi+_zone_lookup.csv",
    "fhvhv_tripdata_2024-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2024-01.parquet"
}

for filename, url in datasets.items():
    file_path = raw_dir / filename
    
    if not file_path.exists():
        print(f"Mengunduh {filename}")
        urllib.request.urlretrieve(url, file_path)
        print(f"-> Unduhan {filename} selesai.")
    else:
        print(f"File {filename} sudah ada, melewati proses unduh.")
        
    with open(file_path, "rb") as f:
        file_hash = hashlib.sha256(f.read()).hexdigest()
        
    print(f"File Name : {filename}")
    print(f"SHA-256   : {file_hash}")

File taxi_zone_lookup.csv sudah ada, melewati proses unduh.
File Name : taxi_zone_lookup.csv
SHA-256   : 1a99e105092230f8620f301edcca7f80d3080642ff404d28ed957d3fa222c8ed
File fhvhv_tripdata_2024-01.parquet sudah ada, melewati proses unduh.
File Name : fhvhv_tripdata_2024-01.parquet
SHA-256   : 9897de352aa52cea36b70348cc6721b8d4494327ce39c85f0dba83d86ecaa098


# 2. Setup Connection & File Paths

In [9]:
import duckdb
import pandas as pd

con = duckdb.connect()

parquet_file = raw_dir / 'fhvhv_tripdata_2024-01.parquet'
zone_file = raw_dir / 'taxi_zone_lookup.csv'

# 3. Quick Preview of Raw Data

In [10]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

df_preview = con.execute(f"SELECT * FROM '{parquet_file}' LIMIT 10").df()
df_preview

,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time,base_passenger_fare,tolls,bcf,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0003,B03404,B03404,2024-01-01 00:21:47,2024-01-01 00:25:06,2024-01-01 00:28:08,2024-01-01 01:05:39,161,158,2.83,2251,45.61,0.00,1.25,4.05,2.75,0.0,0.0,40.18,N,N,N,N,N
1,HV0003,B03404,B03404,2024-01-01 00:10:56,2024-01-01 00:11:08,2024-01-01 00:12:53,2024-01-01 00:20:05,137,79,1.57,432,10.05,0.00,0.28,0.89,2.75,0.0,0.0,6.12,N,N,N,N,N
2,HV0003,B03404,B03404,2024-01-01 00:20:04,2024-01-01 00:21:51,2024-01-01 00:23:05,2024-01-01 00:35:16,79,186,1.98,731,18.07,0.00,0.50,1.60,2.75,0.0,0.0,9.47,N,N,N,N,N
3,HV0003,B03404,B03404,2024-01-01 00:35:46,2024-01-01 00:39:59,2024-01-01 00:41:04,2024-01-01 00:56:34,234,148,1.99,930,17.17,0.00,0.47,1.52,2.75,0.0,0.0,11.35,N,N,N,N,N
4,HV0003,B03404,B03404,2024-01-01 00:48:19,2024-01-01 00:56:23,2024-01-01 00:57:21,2024-01-01 01:10:02,148,97,2.65,761,38.67,0.00,1.06,3.43,2.75,0.0,0.0,28.63,N,N,N,N,N
5,HV0003,B03404,B03404,2024-01-01 00:03:47,2024-01-01 00:05:53,2024-01-01 00:06:15,2024-01-01 00:27:53,255,95,7.02,1298,32.16,0.00,0.88,2.85,0.00,0.0,0.0,24.35,N,N,N,N,Y
6,HV0003,B03404,B03404,2024-01-01 00:22:51,2024-01-01 00:29:17,2024-01-01 00:29:47,2024-01-01 00:50:08,95,212,11.33,1221,45.83,6.94,1.45,4.68,0.00,0.0,0.0,30.98,N,N,N,N,Y
7,HV0003,B03404,B03404,2024-01-01 00:45:34,2024-01-01 00:57:29,2024-01-01 00:57:50,2024-01-01 01:11:27,213,47,3.43,817,23.23,0.00,0.64,2.06,0.00,0.0,0.0,20.73,N,N,N,N,Y
8,HV0003,B03404,B03404,2024-01-01 00:11:51,2024-01-01 00:15:46,2024-01-01 00:16:00,2024-01-01 00:28:13,209,114,1.54,733,15.42,0.00,0.42,1.37,2.75,0.0,0.0,10.40,N,N,N,N,Y
9,HV0003,B03404,B03404,2024-01-01 00:26:48,2024-01-01 00:33:02,2024-01-01 00:33:15,2024-01-01 00:46:39,113,209,1.72,804,13.65,0.00,0.38,1.21,2.75,0.0,0.0,11.38,N,N,N,N,Y


In [11]:
# 1. Cek Jumlah Baris & Struktur Kolom
df_info = con.execute(f"DESCRIBE SELECT * FROM '{parquet_file}'").df()
total_rows = con.execute(f"SELECT COUNT(*) FROM '{parquet_file}'").fetchone()[0]

print(f"Total Baris Data (Jan 2024) : {total_rows:,} baris")
print(f"Total Kolom                : {len(df_info)} kolom\n")

print("--- Struktur Kolom & Tipe Data ---")
for idx, row in df_info.iterrows():
    print(f"  - {row['column_name']:<25}: {row['column_type']}")

print("\n" + "="*60)
print("ENAM AUDIT KUALITAS DATA AWAL (CHECKS 001 - 006)")
print("="*60)

# 2. Audit 1: Cek Rentang Timestamp (Apakah ada data di luar bulan Jan 2024?)
time_range = con.execute(f"""
    SELECT 
        MIN(pickup_datetime) as min_pickup, 
        MAX(pickup_datetime) as max_pickup 
    FROM '{parquet_file}'
""").fetchone()
print(f"1. Audit Timestamp Range : {time_range[0]} s/d {time_range[1]}")

# 3. Audit 2: Cek Jam/Durasi Tidak Logis (Dropoff terjadi SEBELUM Pickup)
invalid_duration = con.execute(f"""
    SELECT COUNT(*) 
    FROM '{parquet_file}' 
    WHERE dropoff_datetime <= pickup_datetime
""").fetchone()[0]
print(f"2. Audit Durasi Negatif  : {invalid_duration} baris bermasalah")

# 4. Audit 3: Cek Anomali Jarak Perjalanan (Jarak <= 0 mil)
zero_miles = con.execute(f"""
    SELECT COUNT(*) 
    FROM '{parquet_file}' 
    WHERE trip_miles <= 0
""").fetchone()[0]
print(f"3. Audit Jarak <= 0 Mil  : {zero_miles:,} baris ({zero_miles/total_rows*100:.2f}%)")

# 5. Audit 4: Cek ID Lokasi Pickup/Dropoff yang Kosong (Null Location)
null_loc = con.execute(f"""
    SELECT 
        COUNT(*) FILTER (WHERE PULocationID IS NULL) as null_pu,
        COUNT(*) FILTER (WHERE DOLocationID IS NULL) as null_do
    FROM '{parquet_file}'
""").fetchone()
print(f"4. Audit Null LocationID : Pickup Null = {null_loc[0]}, Dropoff Null = {null_loc[1]}")

# 6. Audit 5: Integritas Referensi Zone Lookup (Apakah ada ID Lokasi yang tidak ada di CSV lookup?)
invalid_zones = con.execute(f"""
    SELECT COUNT(*) 
    FROM '{parquet_file}' p
    LEFT JOIN '{zone_file}' z ON p.PULocationID = z.LocationID
    WHERE z.LocationID IS NULL
""").fetchone()[0]
print(f"5. Audit Unmapped Zones  : {invalid_zones} ID lokasi tidak terdaftar di Zone Lookup")

Total Baris Data (Jan 2024) : 19,663,930 baris
Total Kolom                : 24 kolom

--- Struktur Kolom & Tipe Data ---
  - hvfhs_license_num        : VARCHAR
  - dispatching_base_num     : VARCHAR
  - originating_base_num     : VARCHAR
  - request_datetime         : TIMESTAMP
  - on_scene_datetime        : TIMESTAMP
  - pickup_datetime          : TIMESTAMP
  - dropoff_datetime         : TIMESTAMP
  - PULocationID             : INTEGER
  - DOLocationID             : INTEGER
  - trip_miles               : DOUBLE
  - trip_time                : BIGINT
  - base_passenger_fare      : DOUBLE
  - tolls                    : DOUBLE
  - bcf                      : DOUBLE
  - sales_tax                : DOUBLE
  - congestion_surcharge     : DOUBLE
  - airport_fee              : DOUBLE
  - tips                     : DOUBLE
  - driver_pay               : DOUBLE
  - shared_request_flag      : VARCHAR
  - shared_match_flag        : VARCHAR
  - access_a_ride_flag       : VARCHAR
  - wav_request_flag   

# 4. Download Remaining Months (February–December)

In [12]:
for month in range(2, 13):
    month_str = f"{month:02d}"
    filename = f"fhvhv_tripdata_2024-{month_str}.parquet"
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{filename}"
    file_path = raw_dir / filename

    if not file_path.exists():
        print(f"[{month_str}/12] Mengunduh {filename}...")
        try:
            urllib.request.urlretrieve(url, file_path)
            print(f"  -> Selesai mengunduh {filename}!")
        except Exception as e:
            print(f"  -> Gagal mengunduh {filename}. Error: {e}")
    else:
        print(f"[{month_str}/12] File {filename} sudah ada, melewati proses unduh.")

[02/12] File fhvhv_tripdata_2024-02.parquet sudah ada, melewati proses unduh.
[03/12] File fhvhv_tripdata_2024-03.parquet sudah ada, melewati proses unduh.
[04/12] File fhvhv_tripdata_2024-04.parquet sudah ada, melewati proses unduh.
[05/12] File fhvhv_tripdata_2024-05.parquet sudah ada, melewati proses unduh.
[06/12] File fhvhv_tripdata_2024-06.parquet sudah ada, melewati proses unduh.
[07/12] File fhvhv_tripdata_2024-07.parquet sudah ada, melewati proses unduh.
[08/12] File fhvhv_tripdata_2024-08.parquet sudah ada, melewati proses unduh.
[09/12] File fhvhv_tripdata_2024-09.parquet sudah ada, melewati proses unduh.
[10/12] File fhvhv_tripdata_2024-10.parquet sudah ada, melewati proses unduh.
[11/12] File fhvhv_tripdata_2024-11.parquet sudah ada, melewati proses unduh.
[12/12] File fhvhv_tripdata_2024-12.parquet sudah ada, melewati proses unduh.


# 5. Save Checksum

In [13]:
checksum_file = raw_dir / 'checksum_log.txt'

with open(checksum_file, 'w') as f_out:
    # Iterasi semua file di folder raw_dir
    for filepath in sorted(raw_dir.iterdir()):
        if filepath.is_file() and filepath.name != 'checksum_log.txt':
            with open(filepath, "rb") as f_in:
                file_hash = hashlib.sha256(f_in.read()).hexdigest()
            
            log_line = f"{filepath.name},{file_hash}\n"
            f_out.write(log_line)
            print(f"Tercatat: {filepath.name}")
            
print(f"\nSemua checksum berhasil disimpan di {checksum_file}")

con.close()

Tercatat: fhvhv_tripdata_2024-01.parquet
Tercatat: fhvhv_tripdata_2024-02.parquet
Tercatat: fhvhv_tripdata_2024-03.parquet
Tercatat: fhvhv_tripdata_2024-04.parquet
Tercatat: fhvhv_tripdata_2024-05.parquet
Tercatat: fhvhv_tripdata_2024-06.parquet
Tercatat: fhvhv_tripdata_2024-07.parquet
Tercatat: fhvhv_tripdata_2024-08.parquet
Tercatat: fhvhv_tripdata_2024-09.parquet
Tercatat: fhvhv_tripdata_2024-10.parquet
Tercatat: fhvhv_tripdata_2024-11.parquet
Tercatat: fhvhv_tripdata_2024-12.parquet
Tercatat: taxi_zone_lookup.csv

Semua checksum berhasil disimpan di ..\data\raw\checksum_log.txt
